In [1]:
from ctypes import c_float, c_int, c_size_t, c_uint, c_uint64, c_void_p, POINTER
import numpy as np, torch, ctypes, json, glob, re, bisect, os
from transformers import AutoTokenizer #FIXME we can use the model's Tokenizer as well, should be 1:1 for downstream actions
from tqdm.auto import tqdm
from pathlib import Path

/Users/bitan/Desktop/aperdiff/python/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NOTE: This notebook just outputs layerwise tensors for the ideas we've put down in Ablation.md, for anyone reading this, it is just experimental, and to ensure that we scientifically approach the layer problem instead of just taking arbitrary middle context layers

In [2]:
cwd  = Path.cwd()
REPO = cwd if (cwd / "core").exists() else cwd.parent

GGUF  = REPO / "core/assets/apertus-v1.5-8b-text-Q5_K_M.gguf" #From https://huggingface.co/andreasmartin/apertus-v1.5-8b-text
DYLIB = REPO / "outputs/lib/libelthur.dylib"

SRC = {
    "train": REPO / "python/dataset/val",        # train on the dataset's dev split
    "val":   REPO / "python/dataset/holdout",    # holdout -> validation / early stop (I renamed this from test in order to )
    "syn":   REPO / "python/dataset/synthetic",  # soft OOD eval
}
OUT = REPO / "python/layerwise"
OUT.mkdir(parents=True, exist_ok=True)

TOKENIZER_REPO = "swiss-ai/Apertus-v1.5-8B"
LAYERS = [14, 15, 16, 17]
LIMIT  = None # HACK rows per language (None = all)

In [3]:
_WS = re.compile(r"\S+")
_CB = ctypes.CFUNCTYPE(None, c_uint, c_int, c_size_t, c_size_t, POINTER(c_float))

def make_engine():
    os.environ["ELTHUR_MODEL"] = str(GGUF)
    lib = ctypes.CDLL(str(DYLIB))
    lib.model_create_ex.restype = c_void_p
    lib.model_create_ex.argtypes = [ctypes.c_char_p, c_int, ctypes.c_double, ctypes.c_double, c_int]
    lib.model_capture.restype = c_int
    lib.model_capture.argtypes = [c_void_p, POINTER(c_int), c_size_t, c_uint64, c_void_p]
    lib.model_free.argtypes = [c_void_p]
    h = lib.model_create_ex(b"apertus", 16, 0.0, 0.9, 40)
    assert h, "model_create_ex failed"
    return lib, h

def capture_layers(lib, h, ids, layers):
    """res_ffn (site 10) for every layer in `layers` -> (len(layers), n_tokens, 4096)."""
    n = len(ids); slot = {l: i for i, l in enumerate(layers)}
    out = np.empty((len(layers), n, 4096), np.float32)
    def cb(s, l, tok, dim, ptr):
        if int(s) == 10:
            i = slot.get(int(l))
            if i is not None and int(tok) < n:
                ctypes.memmove(out[i, int(tok)].ctypes.data, ptr, int(dim) * 4)
    fn = _CB(cb); ci = (c_int * n)(*ids)
    assert lib.model_capture(h, ci, n, 1 << 10, ctypes.cast(fn, c_void_p)) >= 0
    return out

def encode(tok, text):
    e = tok(text, add_special_tokens=True, return_offsets_mapping=True)
    ids = list(e["input_ids"]); offs = list(e["offset_mapping"])
    sp = [(m.start(), m.end()) for m in _WS.finditer(text)]; st = [s for s, _ in sp]; s2w = []
    for a, b in offs:
        if b <= a: s2w.append(-1); continue
        c = b - 1
        i = bisect.bisect_right(st, c) - 1
        s2w.append(i if (i >= 0 and sp[i][0] <= c < sp[i][1]) else -1)
    return ids, s2w, len(sp)

def pool(vec, s2w, nw):
    acc = np.zeros((nw, 4096), np.float32); c = np.zeros((nw, 1), np.float32)
    for t, w in enumerate(s2w):
        if w >= 0: acc[w] += vec[t]; c[w, 0] += 1
    c[c == 0] = 1
    return acc / c

def make_entry(row, lib, h, tok):
    ia, sa, na = encode(tok, row["text_a"]); ib, sb, nb = encode(tok, row["text_b"])
    fa = capture_layers(lib, h, ib + ia, LAYERS)[:, len(ib):, :]   # A sees B
    fb = capture_layers(lib, h, ia + ib, LAYERS)[:, len(ia):, :]   # B sees A
    feats_a = [pool(fa[i], sa, na) for i in range(len(LAYERS))]
    feats_b = [pool(fb[i], sb, nb) for i in range(len(LAYERS))]
    return feats_a, feats_b


In [4]:
def build(role, src):
    counts = {l: 0 for l in LAYERS}
    for f in sorted(glob.glob(str(src / "*.jsonl"))):
        lang = Path(f).stem
        outs = [OUT / f"{role}_{lang}_L{l}.pt" for l in LAYERS]
        if all(p.exists() for p in outs):           # resume: skip finished languages
            print(f"skip {role}/{lang} (exists)"); continue
        rows = [json.loads(x) for x in open(f) if x.strip()]
        if LIMIT: rows = rows[:LIMIT]
        per_layer = {l: [] for l in LAYERS}         # only ONE language in memory
        for r in tqdm(rows, desc=f"{role}/{lang}", ncols=90):
            fa, fb = make_entry(r, lib, h, tok)
            ta = r["text_a"].split(); tb = r["text_b"].split()
            ya = [float(x) for x in r["labels_a"][:len(ta)]]
            yb = [float(x) for x in r["labels_b"][:len(tb)]]
            for i, l in enumerate(LAYERS):
                per_layer[l].append({
                    "id": r["id"], "lang": lang, "layer": l,
                    "tokens_a": ta, "tokens_b": tb, "labels_a": ya, "labels_b": yb,
                    "feats_a": torch.from_numpy(fa[i]).half(),
                    "feats_b": torch.from_numpy(fb[i]).half()})
        for l in LAYERS:
            torch.save(per_layer[l], OUT / f"{role}_{lang}_L{l}.pt")
            counts[l] += len(per_layer[l])
            print(f"{role}_{lang}_L{l}.pt  {len(per_layer[l])} pairs")
        del per_layer
    return counts


In [ ]:
tok = AutoTokenizer.from_pretrained(TOKENIZER_REPO) ## FIXME This shold be exactly the same as the model tokenizer now hoo haa
lib, h = make_engine()
index = {}

[transformers] You are using a model of type `apertus1p5` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


In [6]:
## TODO run for all languages/items
try:
    for role, src in SRC.items():
        if not src.exists():
            print("skip (missing):", src); continue
        counts = build(role, src)
        index[role] = {str(l): counts[l] for l in LAYERS}
finally:
    lib.model_free(h)
(OUT / "index.json").write_text(json.dumps(index, indent=2))
print("done ->", OUT)


train/de: 100%|███████████████████████████████████████| 168/168 [2:13:13<00:00, 47.58s/it]


train_de_L14.pt  168 pairs
train_de_L15.pt  168 pairs
train_de_L16.pt  168 pairs
train_de_L17.pt  168 pairs


train/fr: 100%|███████████████████████████████████████| 168/168 [1:58:37<00:00, 42.37s/it]


train_fr_L14.pt  168 pairs
train_fr_L15.pt  168 pairs
train_fr_L16.pt  168 pairs
train_fr_L17.pt  168 pairs


train/it: 100%|███████████████████████████████████████| 168/168 [2:04:56<00:00, 44.62s/it]


...

syn_de-fr_L14.pt  56 pairs
syn_de-fr_L15.pt  56 pairs
syn_de-fr_L16.pt  56 pairs
syn_de-fr_L17.pt  56 pairs


syn/de-it: 100%|██████████████████████████████████████████| 56/56 [51:49<00:00, 55.52s/it]


syn_de-it_L14.pt  56 pairs
syn_de-it_L15.pt  56 pairs
syn_de-it_L16.pt  56 pairs
syn_de-it_L17.pt  56 pairs


syn/fr-it: 100%|██████████████████████████████████████████| 56/56 [53:19<00:00, 57.13s/it]


syn_fr-it_L14.pt  56 pairs
syn_fr-it_L15.pt  56 pairs
syn_fr-it_L16.pt  56 pairs
syn_fr-it_L17.pt  56 pairs
done -> /Users/bitan/Desktop/aperdiff/python/layerwise
